# Chapter 8 on a GPU: attention-logit growth and QK-norm at larger width

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrendanJamesLynskey/transformer-explainer/blob/main/notebooks/08_attention_logit_growth.ipynb)

The CPU experiment (results.md §8b) shows attention scores growing to 10^4-10^5 at a high
learning rate, and QK-norm keeping them small. Here the same comparison on a 6-layer, width-512
model (about 20M parameters): baseline, QK-norm and z-loss at four learning rates, two seeds,
logging the largest attention score (Wortsman et al. 2023, arXiv:2309.14322).

**GPU:** an L4 or A100. **Expected time:** about 45 min on an A100 (24 runs of 3,000 steps).

**Output:** `attention_logit_growth.json` → `notebooks/results/attention_logit_growth.json`.

Part of *Transformer Decoder Explained*, Part 2 (Training). Generated by `scripts/training/notebooks_build.py`; `SMOKE=1` runs a tiny version on a CPU in seconds.

In [ ]:
import json, math, os, platform, subprocess, sys, time, urllib.request
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

SMOKE = os.environ.get("SMOKE") == "1"   # tiny CPU run (CI and a local check)
TESTED_TORCH = "2.11.0"                        # the version this notebook was written and smoke-tested with
if torch.__version__.split("+")[0] != TESTED_TORCH:
    print(f"note: tested with torch {TESTED_TORCH}, running {torch.__version__} (recorded in the results)")

try:  # Colab: keep checkpoints and results on Drive, so a dropped session resumes
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/transformer-training-runs")
except Exception:
    BASE = Path(os.environ.get("RUNS_DIR", "runs"))
BASE.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() and not SMOKE else "cpu"
print("device:", DEVICE, "| results and checkpoints in", BASE)

In [ ]:
# Data: public-domain books from Project Gutenberg (the gutenberg.pglaf.org
# mirror), byte-level tokens (vocabulary 256), the last 5% for validation.
# Small (about 5 MB): fine for comparing settings, not for a good model.
# All are public domain in the USA; the Project Gutenberg header, footer
# and licence are stripped, as for the site's own corpus (CORPUS.md).
BOOKS = [11, 1342, 2701, 84, 1661, 98, 1080, 345]  # Alice, Pride and Prejudice, Moby Dick, Frankenstein, Sherlock Holmes, A Tale of Two Cities, A Modest Proposal, Dracula


def strip_gutenberg(text: str) -> str:
    a = text.find("*** START OF")
    b = text.find("*** END OF")
    if a != -1:
        text = text[text.find("\n", a) + 1:]
    if b != -1:
        text = text[: text.find("*** END OF")]
    return text


def load_corpus() -> bytes:
    if SMOKE:  # no network: the site's own corpus if it is here, else a stub
        local = Path("public/training/corpus.txt")
        return (local.read_text() if local.exists() else "alice was beginning to get very tired. " * 400).encode()
    cache = BASE / "corpus.bin"
    if cache.exists():
        return cache.read_bytes()
    parts = []
    for n in BOOKS:
        url = f"https://gutenberg.pglaf.org/cache/epub/{n}/pg{n}.txt"
        with urllib.request.urlopen(url, timeout=60) as r:
            parts.append(strip_gutenberg(r.read().decode("utf-8", "ignore")))
    data = "\n\n".join(parts).encode("utf-8")
    cache.write_bytes(data)
    return data


CORPUS = torch.tensor(list(load_corpus()), dtype=torch.long)
cut = int(len(CORPUS) * 0.95)
TRAIN, VAL = CORPUS[:cut], CORPUS[cut:]
print(f"corpus: {len(CORPUS):,} bytes (train {len(TRAIN):,}, validation {len(VAL):,})")


def batch(split, B, S, gen):
    src = TRAIN if split == "train" else VAL
    ix = torch.randint(len(src) - S - 1, (B,), generator=gen)
    x = torch.stack([src[i:i + S] for i in ix])
    y = torch.stack([src[i + 1:i + S + 1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

In [ ]:
# A small GPT (pre-LN by default), with the knobs the chapters measure:
# post-LN, QK-norm, z-loss and muP (Yang et al. 2022: hidden weights at
# std 1/sqrt(fan_in) relative to the base width, Adam LR / m for them,
# attention scores 1/d_k, logits / m), and optional activation
# checkpointing per block.
class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        d, h = c["d"], c["heads"]
        self.c = c
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.o = nn.Linear(d, d, bias=False)
        self.up = nn.Linear(d, 4 * d)
        self.down = nn.Linear(4 * d, d)
        if c.get("qk_norm"):
            self.qn, self.kn = nn.LayerNorm(d // h), nn.LayerNorm(d // h)
        self.max_score = 0.0

    def attn(self, x):
        B, S, D = x.shape
        h = self.c["heads"]
        q, k, v = self.qkv(x).view(B, S, 3, h, D // h).permute(2, 0, 3, 1, 4)
        if self.c.get("qk_norm"):
            q, k = self.qn(q), self.kn(k)
        dk = D // h
        scale = 1 / dk * math.sqrt(self.c["base"] // h) if self.c.get("mup") else 1 / math.sqrt(dk)
        s = (q @ k.transpose(-1, -2)) * scale
        self.max_score = max(self.max_score, float(s.detach().abs().max()))
        mask = torch.triu(torch.ones(S, S, dtype=torch.bool, device=x.device), 1)
        p = torch.softmax(s.masked_fill(mask, float("-inf")).float(), -1).to(v.dtype)
        return self.o((p @ v).transpose(1, 2).reshape(B, S, D))

    def ffn(self, x):
        return self.down(F.gelu(self.up(x), approximate="tanh"))

    def forward(self, x):
        if self.c.get("post"):
            x = self.ln1(x + self.attn(x))
            return self.ln2(x + self.ffn(x))
        x = x + self.attn(self.ln1(x))
        return x + self.ffn(self.ln2(x))


class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.emb = nn.Embedding(256, c["d"])
        self.pos = nn.Embedding(c["seq"], c["d"])
        self.blocks = nn.ModuleList(Block(c) for _ in range(c["layers"]))
        self.lnf = nn.LayerNorm(c["d"])
        m = c["d"] / c["base"] if c.get("mup") else 1.0
        for name, p in self.named_parameters():
            if p.dim() == 2:
                hidden = "blocks" in name
                std = 0.02 / math.sqrt(m) if (hidden and c.get("mup")) else 0.02
                nn.init.normal_(p, 0, std)
            elif name.endswith("bias"):
                nn.init.zeros_(p)
        self.m = m

    def forward(self, x, y):
        S = x.shape[1]
        h = self.emb(x) + self.pos(torch.arange(S, device=x.device))
        for b in self.blocks:
            if self.c.get("checkpoint") and self.training:
                h = torch.utils.checkpoint.checkpoint(b, h, use_reentrant=False)
            else:
                h = b(h)
        logits = self.lnf(h) @ self.emb.weight.T
        if self.c.get("mup"):
            logits = logits / self.m
        logits = logits.float()
        loss = F.cross_entropy(logits.view(-1, 256), y.reshape(-1))
        if self.c.get("z_loss"):
            lse = torch.logsumexp(logits, -1)
            return loss, loss + self.c["z_loss"] * (lse * lse).mean()
        return loss, loss


def param_groups(model, lr, wd):
    """AdamW groups: decay on matrices; muP divides the hidden matrices' LR by m."""
    groups = []
    for name, p in model.named_parameters():
        hidden = p.dim() == 2 and "blocks" in name
        g_lr = lr / model.m if (hidden and model.c.get("mup")) else lr
        groups.append({"params": [p], "lr": g_lr, "base_lr": g_lr, "weight_decay": wd if p.dim() == 2 else 0.0})
    return groups

In [ ]:
def lr_factor(t, total, warmup):
    if t < warmup:
        return (t + 1) / warmup
    return 0.1 + 0.45 * (1 + math.cos(math.pi * (t - warmup) / max(1, total - warmup)))


AMP = {"fp32": None, "bf16": torch.bfloat16, "fp16": torch.float16}


def run(name, c, lr, steps, B, seed, precision="fp32", scaler=False, wd=0.1, warmup=None, eval_every=None,
        fake_fp8=False):
    """One training run; resumes from its checkpoint; returns its log."""
    path = BASE / f"{name}.pt"
    warmup = warmup if warmup is not None else max(1, steps // 20)
    eval_every = eval_every or max(1, steps // 20)
    torch.manual_seed(seed)
    model = GPT(c).to(DEVICE)
    opt = torch.optim.AdamW(param_groups(model, lr, wd), lr=lr, betas=(0.9, 0.95))
    gs = torch.amp.GradScaler("cuda", enabled=scaler and DEVICE == "cuda")
    gen = torch.Generator().manual_seed(seed)
    log = {"name": name, "config": c, "lr": lr, "steps": steps, "batch": B, "seed": seed,
           "precision": precision, "scaler": scaler, "fakeFp8": fake_fp8, "evals": [], "maxScore": [],
           "skipped": 0}
    t0, start = time.time(), 0
    if path.exists():
        ck = torch.load(path, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        gen.set_state(ck["gen"]); log = ck["log"]; start = ck["step"]
        print(f"{name}: resumed at step {start}")
    if fake_fp8:
        install_fake_fp8(model)
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    tokens = 0
    for t in range(start, steps):
        for g in opt.param_groups:
            g["lr"] = g["base_lr"] * lr_factor(t, steps, warmup)
        x, y = batch("train", B, c["seq"], gen)
        with torch.autocast(DEVICE, dtype=AMP[precision], enabled=AMP[precision] is not None):
            loss, total = model(x, y)
        opt.zero_grad(set_to_none=True)
        gs.scale(total).backward()
        gs.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        before = gs.get_scale() if gs.is_enabled() else None
        gs.step(opt); gs.update()
        if before is not None and gs.get_scale() < before:
            log["skipped"] += 1
        tokens += B * c["seq"]
        if not math.isfinite(loss.item()):
            log["divergedAt"] = t
            break
        if (t + 1) % eval_every == 0 or t + 1 == steps:
            model.eval()
            with torch.no_grad(), torch.autocast(DEVICE, dtype=AMP[precision], enabled=AMP[precision] is not None):
                vg = torch.Generator().manual_seed(1234)
                vl = sum(model(*batch("val", B, c["seq"], vg))[0].item() for _ in range(4)) / 4
            model.train()
            log["evals"].append({"step": t + 1, "val": vl, "train": loss.item()})
            log["maxScore"].append({"step": t + 1, "value": max(b.max_score for b in model.blocks)})
            for b in model.blocks:
                b.max_score = 0.0
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "gen": gen.get_state(),
                        "log": log, "step": t + 1}, path)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
        log["peakMemoryBytes"] = torch.cuda.max_memory_allocated()
    log["seconds"] = time.time() - t0
    log["tokensPerSecond"] = tokens / max(1e-9, log["seconds"])
    log["params"] = sum(p.numel() for p in model.parameters())
    return log


def install_fake_fp8(model):
    """FP8 emulation on any GPU: each Linear's input and weight are rounded
    through torch.float8_e4m3fn with a per-tensor scale (amax), as the FP8
    recipes do; the matrix product itself runs in the autocast precision."""
    def q(t):
        s = 448.0 / t.detach().abs().amax().clamp(min=1e-12)
        return (t * s).to(torch.float8_e4m3fn).to(t.dtype) / s
    for mod in model.modules():
        if isinstance(mod, nn.Linear):
            mod.forward = (lambda m: lambda x: F.linear(q(x), q(m.weight), m.bias))(mod)


def metadata():
    meta = {"torch": torch.__version__, "cuda": torch.version.cuda, "python": platform.python_version(),
            "device": DEVICE, "smoke": SMOKE, "testedTorch": TESTED_TORCH,
            "date": time.strftime("%Y-%m-%d %H:%M:%S %Z")}
    if DEVICE == "cuda":
        meta["gpu"] = torch.cuda.get_device_name(0)
        try:
            meta["driver"] = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                                            capture_output=True, text=True).stdout.strip()
        except Exception:
            meta["driver"] = None
    return meta


def save(results, fname):
    out = {"metadata": metadata(), **results}
    p = BASE / fname
    p.write_text(json.dumps(out, indent=1))
    print("wrote", p)
    return p

In [ ]:
LRS = [3e-3, 3e-2] if SMOKE else [1e-3, 3e-3, 1e-2, 3e-2]
SEEDS = [1] if SMOKE else [1, 2]
STEPS, B = (20, 4) if SMOKE else (3000, 32)
c0 = dict(d=64 if SMOKE else 512, heads=4 if SMOKE else 8, layers=2 if SMOKE else 6, seq=32 if SMOKE else 256, base=64)
rows = []
for name, extra in (("baseline", {}), ("QK-norm", {"qk_norm": True}), ("z-loss 1e-4", {"z_loss": 1e-4})):
    for lr in LRS:
        for seed in SEEDS:
            log = run(f"attn-{name}-{lr:g}-{seed}", dict(c0, **extra), lr, STEPS, B, seed,
                      precision="fp32" if SMOKE else "bf16")
            rows.append({"name": name, "lr": lr, "seed": seed, "evals": log["evals"], "maxScore": log["maxScore"],
                         "divergedAt": log.get("divergedAt")})
            print(name, lr, seed, log["evals"][-1] if log["evals"] else "diverged")
save({"experiment": "attention_logit_growth", "steps": STEPS, "batch": B, "rows": rows}, "attention_logit_growth.json")